# 第一版 baseline：近期固定5场 vs 全部历史交锋

**原题。** *For a given matchup between teams, what is more useful to predict results: most recent form, or historical form between the teams. How does this correlate with the betting odds for a given match?*

对一场给定对阵：预测赛果时，更有用的是各队**近期状态**，还是两队之间的**历史交锋**？这些信息和该场**赔率**如何相关？

**问题:** 近期状态、历史交锋与 Elo 不独立。近期结果好的球队往往 Elo 也高、交锋也占优。若分别估计单变量系数，系数会互相污染，无法判断信息来自哪一块。

**解决办法:** 各组保留相同的基础信息 B，再分别加入近期 F、交锋成绩 H，或两者同时加入。比较同一批比赛、同一种模型的样本外损失差。

**输入**

- `data/processed/cleaned.csv`：预测样本（Elo、近 5 场、赔率均完整）
- `data/raw/Matches.csv`：全部有效赛果，用于计算赛前近期与交锋特征

**输出**

后续笔记本还要读取的文件：

- `data/processed/baseline_features.csv`：每场的 B / F / H / Q 特征与时间划分。`time_decay`、`lstm_multitask`、`post_analysis` 用它对齐同一批比赛
- `reports/baseline/test_match_probabilities.csv`：测试集逐场的三项概率与损失。`time_decay` 和 `lstm_multitask` 用它比较，不再重跑本笔记本
- `reports/baseline/models/`：13 个已拟合模型。`post_analysis` 读取系数。文件名表示对照实验，内部仍保存短名 M0–M3、O0–O3、Qcal、Q0–Q3

直接用于报告的文件：

- `reports/baseline/baseline_comparison.png`、`reports/baseline/baseline_comparison.svg`：同一张三面板图。左为 M0–M3 与 Market 的赛果 Log Loss，中为 O0–O3 的赔率 KL，右为 Market、Qcal、Q0–Q3 的赛果 Log Loss
- `reports/baseline/test_result_scores.csv`：测试集平均 Log Loss、Brier、准确率与 KL
- `reports/baseline/paired_loss_differences.csv`：模型两两的损失差及描述性 95% 区间
- `reports/baseline/odds_spearman_correlations.csv`：测试集特征与赔率的 Spearman 相关
- `reports/baseline/baseline.md`：按原题写的结果分析，并引用比较图

## 变量设定

代码中的分组为

- `B = [elo_diff, h2h_count, h2h_missing]`
- `F = [recent_ppg_diff, recent_gd_diff, recent_draw_rate_mean]`
- `H = [h2h_ppg_diff, h2h_gd, h2h_draw_rate]`
- `Q = [q_log_H_D, q_log_A_D]`

`Division` 在除 `Qcal` 外的模型中作训练期独热编码，不计入上述列表。

| 变量 | 定义 |
|---|---|
| `elo_diff` | `HomeElo − AwayElo` |
| `h2h_count` | 严格早于本场的直接交锋场数 |
| `h2h_missing` | `h2h_count = 0` 时为 1，否则为 0 |
| `recent_ppg_diff` | 双方各自最近 5 场场均积分之差；积分按赛果计，胜 3、平 1、负 0 |
| `recent_gd_diff` | 双方各自最近 5 场场均净胜球之差 |
| `recent_draw_rate_mean` | 双方各自最近 5 场平局比例的算术平均 |
| `h2h_ppg_diff` | 全部此前交锋中，本场主队视角的场均积分差；无交锋为 0 |
| `h2h_gd` | 同上视角的场均净胜球；无交锋为 0 |
| `h2h_draw_rate` | 此前交锋的平局比例；无交锋为 0 |
| `q_H`, `q_D`, `q_A` | 由 `OddHome`、`OddDraw`、`OddAway` 的倒数归一化得到 |
| `q_log_H_D`, `q_log_A_D` | `log(q_H / q_D)`、`log(q_A / q_D)`；仅进入 Q 系列输入 |

本场 `FTHome`、`FTAway`、`FTResult` 及射门、半场不进入输入。

## 四组赛果实验

| 模型 | 输入 | 实验目的 |
|---|---|---|
| M0 | `Division`, `elo_diff`, `h2h_count`, `h2h_missing` | 估计基础变量的预测水平 |
| M1 | M0 与 `recent_ppg_diff`, `recent_gd_diff`, `recent_draw_rate_mean` | 估计近期变量的增量影响 |
| M2 | M0 与 `h2h_ppg_diff`, `h2h_gd`, `h2h_draw_rate` | 估计交锋成绩变量的增量影响 |
| M3 | M1 与 M2 的输入并集 | 估计两组变量同时纳入后的增量影响 |

## 模型选择

采用 L2 正则的多分类逻辑回归。类别概率为

$$P(y=k\mid x)=\frac{\exp(w_k^\top x)}{\sum_{j\in\{H,D,A\}}\exp(w_j^\top x)}$$

输入 $x$ 为上表对应的赛前特征：数值变量按训练期标准化，联赛作独热编码。输出为主胜、平局、客胜概率。M 系列的训练目标为本场 `FTResult`。正则强度 $C$ 仅在验证集上从 0.01、0.1、1 中选择。

样本按时间划分：2000—2020 为训练集，2021—2022 为验证集；选定 $C$ 后，以截至 2022 年末的样本重新拟合，并预测 2023—2024。2025 年记录不参与拟合与评价。特征仅使用严格早于比赛日的赛果；同一比赛日先生成全部特征，再更新当日赛果。

## 结果比较

四组模型使用同一测试样本，比赛等权。主指标为 Log Loss，辅以准确率与 Brier 分数，三者均以小者为优。比较采用同一场比赛的损失之差：

- M1 − M0：近期状态的增量
- M2 − M0：历史交锋成绩的增量
- M1 − M2：近期状态与历史交锋的差异
- M3 − M1、M3 − M2：另一项信息的额外增量

差值为负表示前者损失更低。95% 区间由测试期内按年分层的 28 天块 bootstrap 得到；区间包含 0 时，不认定差异稳定。积分差与净胜差共线，不以单个系数符号解释变量作用。

## 赔率实验

三项赔率先转换为隐含概率

$$q_k=\frac{1/\mathrm{Odd}_k}{\sum_j 1/\mathrm{Odd}_j},\quad k\in\{H,D,A\}$$

| 系列 | 输入 | 训练目标 | 实验目的 |
|---|---|---|---|
| O0–O3 | 与 M0–M3 相同，不含赔率 | 隐含概率 $q$ | 估计近期或交锋相对盘口分布的贴近程度，指标为 KL$(q\\|p)$ |
| Market | 无 | 不训练，直接使用 $q$ | 给出赔率对赛果的预测基准 |
| Qcal、Q0–Q3 | 赔率对数比；Q0 起加入 B，并分别加入 F、H | 本场赛果 | 估计给定赔率后 F、H 的增量；Q1、Q2、Q3 与 Q0 比较 |

O1、O2 分别对应近期与历史交锋，比较方式与 M 系列相同。另计算测试集上近期差、交锋差与 $q_H-q_A$ 的 Spearman 相关系数，仅作描述，不作为主要证据。

上述结果限于 Elo、近 5 场与赔率均完整的样本，以及固定 5 场窗口、全部交锋与逻辑回归。2023—2024 已在此前分析中使用，本结果为后续比较。预测损失的下降不构成投注依据。

## 1. 读取预处理后的数据cleaned.csv
**问题:** 笔记本可能从项目根目录或 `notebooks` 目录运行。原始表中的空值写法也不统一。

**解决办法:** 先定位含 `data/raw/Matches.csv` 的目录，统一缺失标记和联赛到国家的映射，再读入 `cleaned.csv`。

In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import unicodedata
import warnings
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from threadpoolctl import threadpool_limits

ROOT = Path.cwd()
if not (ROOT / "data/raw/Matches.csv").exists():
    ROOT = ROOT.parent

CORE = ["Division", "MatchDate", "HomeTeam", "AwayTeam", "FTHome", "FTAway", "FTResult"]
MISSING = {"", "nan", "na", "n/a", "null", "none"}
COUNTRY = {d: d for d in "ARG AUT BRA CHN DEN FIN IRL JAP MEX NOR POL ROM RUS SUI SWE USA".split()}
COUNTRY.update({d: c for c, ds in {
    "ENG": "E0 E1 E2 E3 EC", "SCO": "SC0 SC1 SC2 SC3", "GER": "D1 D2",
    "FRA": "F1 F2", "ITA": "I1 I2", "ESP": "SP1 SP2", "BEL": "B1",
    "GRE": "G1", "NED": "N1", "POR": "P1", "TUR": "T1",
}.items() for d in ds.split()})
SEED, C_GRID = 20261028, [0.01, 0.1, 1.0]
TRAIN_END, VALID_END, TEST_END = "2020-12-31", "2022-12-31", "2024-12-31"
CLASSES, EPS = ["H", "D", "A"], 1e-15
OUT = ROOT / "reports/baseline"
(OUT / "models").mkdir(parents=True, exist_ok=True)

def norm(x):
    return " ".join(unicodedata.normalize("NFKC", str(x)).split()) if pd.notna(x) else x

def read_raw(path):
    frame = pd.read_csv(path, dtype=str, keep_default_na=False)
    for col in frame:
        frame.loc[frame[col].str.strip().str.lower().isin(MISSING), col] = np.nan
    return frame

cleaned = pd.read_csv(ROOT / "data/processed/cleaned.csv", parse_dates=["MatchDate"])
print("cleaned:", cleaned.shape)

## 2. 建立完整赛果历史池

**问题:** `cleaned.csv` 只保留同时拥有 Elo、完整近 5 场和赔率的比赛。某些球队最近 5 场，或两队此前的交锋，可能缺 Elo 或赔率，因而进不了 `cleaned.csv`。若近况和交锋只在这份表里回看，已知赛果会被漏掉。升降级后的比赛若按联赛拆开，也会把同一支球队的历史切断。

**解决办法:** 预测样本仍只从 `cleaned.csv` 选取。计算近 5 场和交锋时，改读 `Matches.csv` 中全部核心信息有效的比赛：日期、联赛、双方队名和全场比分齐全，且赛果与比分一致。Elo 或赔率缺失不影响进入历史。同一国家内，甲级与乙级连续计算。队名只做 Unicode 和空白规范化，不合并相似队名。同一天先生成全部特征，再把当日赛果写入历史。

In [ ]:
def prepare_matches(raw):
    df = raw.copy()
    df.insert(0, "SourceRow", np.arange(2, len(df) + 2))
    for col in ["Division", "HomeTeam", "AwayTeam"]:
        df[col] = df[col].map(norm)
    df["MatchDate"] = pd.to_datetime(df.MatchDate, format="%Y-%m-%d", errors="coerce")
    df["FTHome"] = pd.to_numeric(df.FTHome, errors="coerce")
    df["FTAway"] = pd.to_numeric(df.FTAway, errors="coerce")
    scores = df[["FTHome", "FTAway"]]
    expected = np.where(df.FTHome > df.FTAway, "H", np.where(df.FTHome < df.FTAway, "A", "D"))
    ok = (df[CORE].notna().all(axis=1) & df.MatchDate.notna()
          & np.isfinite(scores).all(axis=1) & scores.ge(0).all(axis=1) & scores.mod(1).eq(0).all(axis=1)
          & df.FTResult.eq(expected) & df.HomeTeam.ne(df.AwayTeam))
    df = df.loc[ok].copy()
    df["Country"] = df.Division.map(COUNTRY)
    if df.Country.isna().any():
        raise ValueError("Unknown division code.")
    df[["FTHome", "FTAway"]] = df[["FTHome", "FTAway"]].astype(int)
    if df.duplicated(["Division", "MatchDate", "HomeTeam", "AwayTeam"]).any():
        raise ValueError("Duplicate matches.")
    df = df.sort_values(["MatchDate", "SourceRow"]).reset_index(drop=True)
    apps = pd.concat([
        df[["Country", "MatchDate", "HomeTeam"]].rename(columns={"HomeTeam": "Team"}),
        df[["Country", "MatchDate", "AwayTeam"]].rename(columns={"AwayTeam": "Team"})])
    if apps.duplicated(["Country", "MatchDate", "Team"]).any():
        raise ValueError("A team has more than one match on the same day.")
    return df

valid_history = prepare_matches(read_raw(ROOT / "data/raw/Matches.csv"))
print("valid history:", len(valid_history))

## 3. 明确每个输入特征

**问题:** 近 5 场、交锋和赔率的原始尺度不同，主客方向也不一致。直接把比分或赔率原值送进模型，无法比较近期和交锋哪一块有用。

**解决办法:** 全部改成赛前的差或比例。近 5 场用各队自己的视角，交锋用本场主队的视角。没有交锋时，H 三项记 0，并用 `h2h_missing` 标出。赔率先变成概率，再取相对平局的对数比。

| 组 | 列名 | 含义 |
|---|---|---|
| B | Division | 联赛，训练期独热编码 |
| B | elo_diff | 当前主队Elo减客队Elo |
| B | h2h_count | 此前全部直接交锋次数 |
| B | h2h_missing | 没有交锋为1，否则0 |
| F | recent_ppg_diff | 双方各自最近5场场均积分之差 |
| F | recent_gd_diff | 双方各自最近5场场均净胜球之差 |
| F | recent_draw_rate_mean | 双方各自最近5场平局比例的平均值 |
| H | h2h_ppg_diff | 直接交锋中，本场主队积分减本场客队积分的平均值 |
| H | h2h_gd | 直接交锋中，本场主队进球减失球的平均值；不重复做一次差 |
| H | h2h_draw_rate | 直接交锋中平局的比例 |
| Q | q_log_H_D、q_log_A_D | 归一化赔率概率的两个对数比，仅用于包含赔率输入的模型 |

无交锋时，H三项为0，B里的无交锋标记为1。当前比赛的比分、赛果、射门和半场等不进入X。

近期特征的主客队是当前比赛双方；每场历史成绩都转换到该球队视角。交锋特征始终转换到本场主队视角。同一比赛日先用已有历史生成全部特征，再把当日赛果写入历史。

近期窗口为双方各自严格早于比赛日的最近 5 场，对手不限。积分按该队赛果计，胜 3、平 1、负 0。$d_i$ 表示该场是否平局。

$$
\mathrm{recent\_ppg\_diff}=\bar p_H-\bar p_A,\qquad
\bar p=\frac{1}{5}\sum_{i=1}^{5}\mathrm{pts}_i
$$

$$
\mathrm{recent\_gd\_diff}=\bar g_H-\bar g_A,\qquad
\mathrm{recent\_draw\_rate\_mean}=\frac{\bar d_H+\bar d_A}{2}
$$

交锋为两队严格早于比赛日的全部直接比赛，场数 $n=\mathrm{h2h\_count}$。积分差与净胜球取本场主队视角。$n=0$ 时三项为 0，且 $\mathrm{h2h\_missing}=1$。

$$
\mathrm{h2h\_ppg\_diff}=\frac{1}{n}\sum_{t=1}^{n}(\mathrm{pts}_{\mathrm{Home},t}-\mathrm{pts}_{\mathrm{Away},t})
$$

$$
\mathrm{h2h\_gd}=\frac{1}{n}\sum_{t=1}^{n}(\mathrm{GF}_{\mathrm{Home},t}-\mathrm{GA}_{\mathrm{Home},t}),\qquad
\mathrm{h2h\_draw\_rate}=\frac{1}{n}\sum_{t=1}^{n}d_t
$$

$$
\mathrm{elo\_diff}=\mathrm{HomeElo}-\mathrm{AwayElo}
$$

$$
q_k=\frac{1/\mathrm{Odd}_k}{\sum_{j\in\{H,D,A\}}1/\mathrm{Odd}_j},\qquad
\mathrm{q\_log\_H\_D}=\log\frac{q_H}{q_D},\qquad
\mathrm{q\_log\_A\_D}=\log\frac{q_A}{q_D}
$$

本节写出 `data/processed/baseline_features.csv`。

In [ ]:
def make_history_features(history_frame, wanted_ids):
    recent = defaultdict(lambda: deque(maxlen=5))
    pairs = {}
    rows = []
    for day, batch in history_frame.groupby("MatchDate", sort=True):
        for r in batch.itertuples():
            if r.SourceRow not in wanted_ids:
                continue
            home_recent = list(recent[(r.Country, r.HomeTeam)])
            away_recent = list(recent[(r.Country, r.AwayTeam)])
            means = []
            for records in (home_recent, away_recent):
                if len(records) == 5:
                    means.append((sum(x[1] for x in records) / 5, sum(x[2] for x in records) / 5, sum(x[3] for x in records) / 5))
                else:
                    means.append((np.nan, np.nan, np.nan))
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            n, points_sum, gd_sum, draws, _ = pairs.get((r.Country, first, second), (0, 0, 0, 0, None))
            sign = 1 if r.HomeTeam == first else -1
            rows.append({
                "SourceRow": r.SourceRow,
                "recent_ppg_diff": means[0][0] - means[1][0],
                "recent_gd_diff": means[0][1] - means[1][1],
                "recent_draw_rate_mean": (means[0][2] + means[1][2]) / 2,
                "home_recent_points_sum": None if len(home_recent) < 5 else sum(x[1] for x in home_recent),
                "away_recent_points_sum": None if len(away_recent) < 5 else sum(x[1] for x in away_recent),
                "h2h_count": n, "h2h_missing": int(n == 0),
                "h2h_ppg_diff": sign * points_sum / n if n else 0.0,
                "h2h_gd": sign * gd_sum / n if n else 0.0,
                "h2h_draw_rate": draws / n if n else 0.0,
            })
        for r in batch.itertuples():
            home_points = 3 if r.FTResult == "H" else 1 if r.FTResult == "D" else 0
            away_points = 3 if r.FTResult == "A" else 1 if r.FTResult == "D" else 0
            gd, draw = r.FTHome - r.FTAway, int(r.FTResult == "D")
            recent[(r.Country, r.HomeTeam)].append((day, home_points, gd, draw))
            recent[(r.Country, r.AwayTeam)].append((day, away_points, -gd, draw))
            first, second = sorted([r.HomeTeam, r.AwayTeam])
            key, sign = (r.Country, first, second), 1 if r.HomeTeam == first else -1
            n, ps, gs, ds, _ = pairs.get(key, (0, 0, 0, 0, None))
            pairs[key] = (n + 1, ps + sign * (home_points - away_points), gs + sign * gd, ds + draw, day)
    return pd.DataFrame(rows)

history = make_history_features(valid_history, set(cleaned.SourceRow))
joined = cleaned.merge(history, on="SourceRow", validate="one_to_one").sort_values(["MatchDate", "SourceRow"]).reset_index(drop=True)
joined["elo_diff"] = joined.HomeElo - joined.AwayElo
joined["year"] = joined.MatchDate.dt.year
joined["y"] = joined.FTResult.map({"H": 0, "D": 1, "A": 2}).astype(int)
q = 1 / joined[["OddHome", "OddDraw", "OddAway"]].to_numpy(float)
q /= q.sum(axis=1, keepdims=True)
joined[["q_H", "q_D", "q_A"]] = q
joined["q_log_H_D"] = np.log(q[:, 0] / q[:, 1])
joined["q_log_A_D"] = np.log(q[:, 2] / q[:, 1])
B = ["elo_diff", "h2h_count", "h2h_missing"]
F = ["recent_ppg_diff", "recent_gd_diff", "recent_draw_rate_mean"]
H = ["h2h_ppg_diff", "h2h_gd", "h2h_draw_rate"]
Q = ["q_log_H_D", "q_log_A_D"]
keep = ["SourceRow", "MatchDate", "year", "Country", "Division", "HomeTeam", "AwayTeam", "FTResult", "y"]
features = joined[keep + B + F + H + Q + ["q_H", "q_D", "q_A", "home_recent_points_sum", "away_recent_points_sum"]].copy()
features["split"] = np.select(
    [features.MatchDate.le(TRAIN_END), features.MatchDate.le(VALID_END), features.MatchDate.le(TEST_END)],
    ["train", "validation", "test"], default="excluded_after_2024")
features.to_csv(ROOT / "data/processed/baseline_features.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print("features:", features.shape)
display(features[["MatchDate", "HomeTeam", "AwayTeam"] + B + F + H].head())

## 4. 根据时间划分数据集测试集和验证集

**问题:** 若随机划分训练和测试，较晚的比赛会进入训练，模型会看到未来赛果。

**解决办法:** 按比赛日期划分。2000—2020 为训练集，2021—2022 为验证集，2023—2024 为测试集。2025 及以后只保留在特征表中，不进入拟合或评分。

按比赛日期划分训练、验证和测试。2025 及以后只保留在特征表中，不进入拟合或评分。

In [ ]:
train = features.loc[features.split.eq("train")].reset_index(drop=True)
validation = features.loc[features.split.eq("validation")].reset_index(drop=True)
test = features.loc[features.split.eq("test")].reset_index(drop=True)
split_table = features.groupby("split", sort=False).agg(
    matches=("SourceRow", "size"), first_date=("MatchDate", "min"),
    last_date=("MatchDate", "max"), divisions=("Division", "nunique")).reset_index()
display(split_table)

## 5. 模型与评价函数

**问题:** 近期、交锋、Elo 和赔率互相相关。只训练一个包含全部变量的模型，看不出多出来的预测力来自哪一块。

**解决办法:** 训练 13 个相同类型的 L2 多分类逻辑回归。每场比赛先构成一行，各模型只读取自己的列。M 系列预测赛果，O 系列拟合赔率分布，Q 系列在已知赔率后再预测赛果。Market 不训练，直接输出赔率概率。

一场比赛的输出是三项概率，顺序固定为 H、D、A：

$$
p_k=\frac{\exp(w_k^\top x)}{\sum_{j\in\{H,D,A\}}\exp(w_j^\top x)},\qquad k\in\{H,D,A\},\qquad p_H+p_D+p_A=1
$$

数值列在进入 $x$ 之前按训练集标准化，$z=(u-\mu)/\sigma$。`Division` 按训练集做独热编码；验证集或测试集中未见过的联赛编码为全 0。Qcal 不含 `Division`。每场比赛等权，不使用类别平衡权重。

| 模型 | 输入 $x$ | 训练标签 | 输出 |
|---|---|---|---|
| M0 | `Division`, B | `FTResult` | $(p_H,p_D,p_A)$ |
| M1 | `Division`, B, F | `FTResult` | $(p_H,p_D,p_A)$ |
| M2 | `Division`, B, H | `FTResult` | $(p_H,p_D,p_A)$ |
| M3 | `Division`, B, F, H | `FTResult` | $(p_H,p_D,p_A)$ |
| O0–O3 | 与 M0–M3 相同，不含赔率 | $q=(q_H,q_D,q_A)$ | $(p_H,p_D,p_A)$ |
| Market | 无 | 无 | $(q_H,q_D,q_A)$ |
| Qcal | Q | `FTResult` | $(p_H,p_D,p_A)$ |
| Q0 | `Division`, Q, B | `FTResult` | $(p_H,p_D,p_A)$ |
| Q1 | `Division`, Q, B, F | `FTResult` | $(p_H,p_D,p_A)$ |
| Q2 | `Division`, Q, B, H | `FTResult` | $(p_H,p_D,p_A)$ |
| Q3 | `Division`, Q, B, F, H | `FTResult` | $(p_H,p_D,p_A)$ |

例。一场比赛的原始值为 `Division=E0`，`B=(150, 2, 0)`，`F=(1.0, 1.2, 0.3)`，`H=(1.5, 1.0, 0.5)`，`Q=(0.405, -0.693)`，对应 $q=(0.500, 0.333, 0.167)$。M0 的输入是联赛 E0 与 `(150, 2, 0)`；M1 在其后追加 `(1.0, 1.2, 0.3)`。两个模型都输出三个概率，数值因输入不同而不同。

O 系列把每场训练比赛复制为三行，标签依次为 H、D、A，权重依次为 $q_H,q_D,q_A$。预处理器仍在未复制的训练比赛上拟合。输入中不包含赔率本身，只有拟合目标是赔率。最小化

$$
L_O=-\sum_{k\in\{H,D,A\}} q_k\log p_k
$$

M 系列与 Q 系列的单场训练损失为 $-\log p_y$，$y$ 为本场 `FTResult`。

评价时 $y$ 为真实赛果。Log Loss 与 Brier 越小越好。O 系列另以 $\mathrm{KL}(q\Vert p)$ 衡量预测分布与赔率分布的距离。

$$
\mathrm{LogLoss}=-\log p_y
$$

$$
\mathrm{Brier}=\sum_{k\in\{H,D,A\}}(p_k-\mathbf{1}_{y=k})^2
$$

$$
\mathrm{Accuracy}=\mathbf{1}\{\arg\max_k p_k=y\}
$$

$$
\mathrm{KL}(q\Vert p)=\sum_{k\in\{H,D,A\}} q_k\log\frac{q_k}{p_k}
$$

若真实赛果为主胜，预测为 $(0.50, 0.30, 0.20)$，该场 Log Loss 为 $-\log 0.50$，Brier 为 $(0.50-1)^2+(0.30-0)^2+(0.20-0)^2$。

接口依据：[scikit-learn LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)。当前 1.9 接口用 `l1_ratio=0` 指定 L2，避免已弃用的 `penalty` 参数。

In [ ]:
def normalize_probabilities(p):
    p = np.clip(np.asarray(p, dtype=float), EPS, 1)
    return p / p.sum(axis=1, keepdims=True)

def match_losses(frame, probabilities):
    p = normalize_probabilities(probabilities)
    target_q = frame[["q_H", "q_D", "q_A"]].to_numpy(float)
    y = frame.y.to_numpy(int)
    return {
        "log_loss": -np.log(p[np.arange(len(frame)), y]),
        "brier": np.square(p - np.eye(3)[y]).sum(axis=1),
        "accuracy": (p.argmax(axis=1) == y).astype(float),
        "kl_to_odds": np.sum(target_q * (np.log(target_q) - np.log(p)), axis=1),
        "odds_probability_mae": np.abs(target_q - p).mean(axis=1),
    }

def fit_classifier(X, frame, target, C):
    if target == "odds":
        train_X = np.repeat(X, 3, axis=0)
        train_y = np.tile(np.arange(3), len(frame))
        weights = frame[["q_H", "q_D", "q_A"]].to_numpy(float).ravel()
    else:
        train_X, train_y, weights = X, frame.y.to_numpy(int), None
    for max_iter in (2000, 10000):
        model = LogisticRegression(C=C, l1_ratio=0, solver="lbfgs", tol=1e-7, max_iter=max_iter, random_state=SEED)
        with warnings.catch_warnings(record=True) as seen, threadpool_limits(limits=1):
            warnings.simplefilter("always", ConvergenceWarning)
            model.fit(train_X, train_y, sample_weight=weights)
        if not any(issubclass(w.category, ConvergenceWarning) for w in seen) and model.n_iter_.max() < max_iter:
            return model
    raise RuntimeError("Logistic regression did not converge.")

def make_preprocessor(columns, include_division=True):
    transformers = [("numeric", StandardScaler(), columns)]
    if include_division:
        transformers.append(("division", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["Division"]))
    return ColumnTransformer(transformers, remainder="drop", sparse_threshold=0)

specs = {}
for suffix, extra in [("0", []), ("1", F), ("2", H), ("3", F + H)]:
    specs["M" + suffix] = {"group": "outcome", "target": "outcome", "columns": B + extra, "division": True}
    specs["O" + suffix] = {"group": "odds_fit", "target": "odds", "columns": B + extra, "division": True}
    specs["Q" + suffix] = {"group": "odds_increment", "target": "outcome", "columns": B + Q + extra, "division": True}
specs["Qcal"] = {"group": "odds_calibration", "target": "outcome", "columns": Q, "division": False}
model_order = ["M0", "M1", "M2", "M3", "O0", "O1", "O2", "O3", "Qcal", "Q0", "Q1", "Q2", "Q3"]
MODEL_FILE = {
    "M0": "M0_outcome_league_elo.joblib",
    "M1": "M1_outcome_plus_recent_form.joblib",
    "M2": "M2_outcome_plus_head_to_head.joblib",
    "M3": "M3_outcome_plus_form_and_h2h.joblib",
    "O0": "O0_fit_odds_league_elo.joblib",
    "O1": "O1_fit_odds_plus_recent_form.joblib",
    "O2": "O2_fit_odds_plus_head_to_head.joblib",
    "O3": "O3_fit_odds_plus_form_and_h2h.joblib",
    "Qcal": "Qcal_outcome_odds_only.joblib",
    "Q0": "Q0_outcome_odds_plus_league_elo.joblib",
    "Q1": "Q1_outcome_odds_plus_recent_form.joblib",
    "Q2": "Q2_outcome_odds_plus_head_to_head.joblib",
    "Q3": "Q3_outcome_odds_plus_form_and_h2h.joblib",
}
inputs_table = pd.DataFrame([
    {"model": name, "group": specs[name]["group"], "target": specs[name]["target"],
     "input_columns": ", ".join((["Division"] if specs[name]["division"] else []) + specs[name]["columns"])}
    for name in model_order])
display(inputs_table)

## 6. 在验证集使用不同的正则化强度

**问题:** 若在测试集上选择正则强度 $C$，测试结果会偏乐观。$C$ 过小会欠拟合，过大则系数容易受少数比赛影响。

**解决办法:** $C$ 只从 0.01、0.1、1 中选择，并且只使用验证集。赛果模型按验证 Log Loss 选择，赔率拟合按验证 KL 选择。分数相差不超过 1e-8 时选择较小的 $C$。

`l1_ratio=0` 表示纯 L2，截距不惩罚。$C$ 是正则强度的倒数，lbfgs 最小化

$$
J(w)=\frac{1}{S}\sum_i s_i(-\log p_{y_i})+\frac{1}{2CS}\|w\|_2^2
$$

赛果模型每场 $s_i=1$，$S$ 为训练场数。O 系列按复制后的三行计，权重为 $q_k$，一场权重和为 1，故 $S$ 仍等于训练场数。$C$ 越小，$\|w\|_2^2$ 前的系数越大。

近期窗口始终为 5，不调整交锋范围。这一节只用训练和验证数据。

选中的 C 保存在对应的 `.joblib` 里，不再另存验证分数表。

In [ ]:
validation_rows, selected = [], {}
for name in model_order:
    spec = specs[name]
    prep = make_preprocessor(spec["columns"], spec["division"])
    X_train, X_validation = prep.fit_transform(train), prep.transform(validation)
    candidates = []
    for C in C_GRID:
        estimator = fit_classifier(X_train, train, spec["target"], C)
        with threadpool_limits(limits=1):
            metrics = {key: float(value.mean()) for key, value in match_losses(validation, estimator.predict_proba(X_validation)).items()}
        primary = metrics["kl_to_odds" if spec["target"] == "odds" else "log_loss"]
        row = {"model": name, "C": C, "primary_score": primary, "iterations": int(estimator.n_iter_.max()), **metrics}
        validation_rows.append(row)
        candidates.append(row)
    best_score = min(row["primary_score"] for row in candidates)
    best = min((row for row in candidates if row["primary_score"] <= best_score + 1e-8), key=lambda row: row["C"])
    selected[name] = {"C": best["C"], "validation_primary_score": best["primary_score"]}
    print(name, "selected C=", best["C"], "validation score=", round(best["primary_score"], 6), flush=True)
validation_scores = pd.DataFrame(validation_rows)
display(validation_scores[["model", "C", "primary_score", "iterations"]])

## 7. 最终训练与测试

**问题:** 验证集已经用来选择 $C$。若仍用只在训练集上拟合的模型评价测试，就没有用上验证期的比赛；若在测试期继续调参，测试就不再独立。

**解决办法:** 选定 $C$ 后，把训练集和验证集合并，重新拟合预处理器和模型，再对 2023—2024 预测一次。测试期间不改变 $C$，也不重新拟合。

赛前历史可以包含此前已经结束的测试期比赛。

Market是原始归一化赔率概率，不训练。Qcal只用两个赔率对数比，衡量简单校准的效果。Q0—Q3分别使用Q+B、Q+B+F、Q+B+H、Q+B+F+H；历史增益应比较Q1／Q2／Q3与Q0。

所有概率按H、D、A顺序保存。O系列输出用于拟合赔率，其真实赛果评分只是辅助结果。

本节写出 `reports/baseline/test_match_probabilities.csv`、`reports/baseline/test_result_scores.csv`，以及 `reports/baseline/models/` 中的 13 个文件：

| 文件 | 对照 |
|---|---|
| `M0_outcome_league_elo.joblib` | 赛果：联赛与 Elo 等基础信息 B |
| `M1_outcome_plus_recent_form.joblib` | 赛果：B + 近 5 场 F |
| `M2_outcome_plus_head_to_head.joblib` | 赛果：B + 全部交锋 H |
| `M3_outcome_plus_form_and_h2h.joblib` | 赛果：B + F + H |
| `O0_fit_odds_league_elo.joblib` | 拟合赔率：输入同 M0 |
| `O1_fit_odds_plus_recent_form.joblib` | 拟合赔率：输入同 M1 |
| `O2_fit_odds_plus_head_to_head.joblib` | 拟合赔率：输入同 M2 |
| `O3_fit_odds_plus_form_and_h2h.joblib` | 拟合赔率：输入同 M3 |
| `Qcal_outcome_odds_only.joblib` | 赛果：只用赔率对数比 |
| `Q0_outcome_odds_plus_league_elo.joblib` | 赛果：赔率 + B |
| `Q1_outcome_odds_plus_recent_form.joblib` | 赛果：赔率 + B + F |
| `Q2_outcome_odds_plus_head_to_head.joblib` | 赛果：赔率 + B + H |
| `Q3_outcome_odds_plus_form_and_h2h.joblib` | 赛果：赔率 + B + F + H |

In [ ]:
final_train = pd.concat([train, validation], ignore_index=True)
predictions = {}
for name in model_order:
    spec = specs[name]
    prep = make_preprocessor(spec["columns"], spec["division"])
    X_final, X_test = prep.fit_transform(final_train), prep.transform(test)
    estimator = fit_classifier(X_final, final_train, spec["target"], selected[name]["C"])
    with threadpool_limits(limits=1):
        predictions[name] = normalize_probabilities(estimator.predict_proba(X_test))
    joblib.dump({"name": name, "preprocessor": prep, "model": estimator, "classes": CLASSES, "spec": spec,
                 "C": selected[name]["C"], "train_end": VALID_END}, OUT / "models" / MODEL_FILE[name])
    print(name, "test log loss=", round(match_losses(test, predictions[name])["log_loss"].mean(), 6), flush=True)
predictions["Market"] = test[["q_H", "q_D", "q_A"]].to_numpy(float)
loss_arrays = {name: match_losses(test, p) for name, p in predictions.items()}
scores = pd.DataFrame([
    {"model": name, "group": specs[name]["group"] if name != "Market" else "raw_odds", "matches": len(test),
     **{key: float(value.mean()) for key, value in loss_arrays[name].items()}}
    for name in model_order + ["Market"]])
rows = []
for name in model_order + ["Market"]:
    part = test[["SourceRow", "MatchDate", "Division", "HomeTeam", "AwayTeam", "FTResult", "y", "q_H", "q_D", "q_A"]].copy()
    part.insert(0, "model", name)
    part[["p_H", "p_D", "p_A"]] = predictions[name]
    part["log_loss"] = loss_arrays[name]["log_loss"]
    part["kl_to_odds"] = loss_arrays[name]["kl_to_odds"]
    rows.append(part)
pd.concat(rows, ignore_index=True).to_csv(OUT / "test_match_probabilities.csv", index=False)
scores.to_csv(OUT / "test_result_scores.csv", index=False)
display(scores)

## 8. 配对比较：差异是否稳定

**问题:** 两个模型的平均 Log Loss 可能只差千分之几。比赛在时间上相关，若把每场当成独立样本，很小的差距也会被说成稳定优势。

**解决办法:** 对同一场比赛的损失相减，负数表示前一个模型更好。在每个测试年内按 28 天分块，有放回抽取 1000 次，得到描述性 95% 区间。区间包含 0 时，不认定差异稳定。

区间只描述当前测试比赛的波动，不包含重新训练和参数选择的不确定性，也不是对多项比较同时成立的 95% 保证。

本节写出 `reports/baseline/paired_loss_differences.csv`。分析写入 `reports/baseline/baseline.md`。

In [ ]:
rng = np.random.default_rng(SEED)
block_weights = []
blocks = []
for year in sorted(test.year.unique()):
    ids = np.flatnonzero(test.year.eq(year).to_numpy())
    day_index = (test.loc[ids,'MatchDate'] - pd.Timestamp(f'{int(year)}-01-01')).dt.days.to_numpy()
    group_id = day_index // 28
    year_blocks = [ids[group_id==group] for group in np.unique(group_id)]
    n_blocks = len(year_blocks)
    draws = rng.integers(0,n_blocks,size=(1000,n_blocks))
    counts = np.zeros((1000,n_blocks),dtype=int)
    for j in range(n_blocks):
        counts[:,j] = (draws==j).sum(axis=1)
    block_weights.append(counts)
    blocks.extend(year_blocks)
weights = np.concatenate(block_weights,axis=1)
block_sizes = np.array([len(ids) for ids in blocks])
bootstrap_denominator = weights @ block_sizes

def paired_difference(left, right, metric):
    difference = loss_arrays[left][metric] - loss_arrays[right][metric]
    sums = np.array([difference[ids].sum() for ids in blocks])
    bootstrap = weights @ sums / bootstrap_denominator
    low, high = np.quantile(bootstrap,[.025,.975])
    return {'left':left,'right':right,'metric':metric,'difference':float(difference.mean()),
            'ci_low':float(low),'ci_high':float(high),'ci_includes_zero':bool(low<=0<=high)}

comparisons = [
    ('M1','M0','log_loss'),('M2','M0','log_loss'),('M3','M0','log_loss'),
    ('M1','M2','log_loss'),('M3','M1','log_loss'),('M3','M2','log_loss'),
    ('O1','O0','kl_to_odds'),('O2','O0','kl_to_odds'),('O3','O0','kl_to_odds'),
    ('O1','O2','kl_to_odds'),('O3','O1','kl_to_odds'),('O3','O2','kl_to_odds'),
    ('Qcal','Market','log_loss'),('Q0','Qcal','log_loss'),
    ('Q1','Q0','log_loss'),('Q2','Q0','log_loss'),('Q3','Q0','log_loss'),
]
paired = pd.DataFrame([paired_difference(*args) for args in comparisons])
paired.to_csv(OUT / "paired_loss_differences.csv", index=False)
display(paired)

## 9. 历史统计与赔率的相关性

**问题:** 赛果损失下降只说明预测更接近真实结果，不能说明近期或交锋是否已经体现在赔率中。

**解决办法:** 仅在测试集上，方向性特征与 $q_H-q_A$ 计算 Spearman 相关系数，平局比例与 $q_D$ 计算相关系数，并按年份分开。这些相关没有控制 Elo 和联赛，不能据此判断博彩公司使用了这些指标。

O0—O3 在相同基础信息上比较谁更接近赔率，作为相关分析之外的补充。

本节写出 `reports/baseline/odds_spearman_correlations.csv`。分析写入 `reports/baseline/baseline.md`。

同时保存一张三面板图，文件为 `reports/baseline/baseline_comparison.png` 与 `reports/baseline/baseline_comparison.svg`。左面板是 M0–M3 与 Market 的赛果 Log Loss，中面板是 O0–O3 的赔率 KL，右面板是 Market、Qcal、Q0–Q3 的赛果 Log Loss。三张图都使用连续纵轴，数值写在类别名下方。

In [ ]:
correlation_pairs = [(col,'home_minus_away') for col in ['recent_ppg_diff','recent_gd_diff','h2h_ppg_diff','h2h_gd']]
correlation_pairs += [('recent_draw_rate_mean','draw_probability'),('h2h_draw_rate','draw_probability')]
correlation_rows = []
for label, frame in [('all_test',test)]+[(str(int(year)),test.loc[test.year.eq(year)]) for year in sorted(test.year.unique())]:
    for col, target_name in correlation_pairs:
        target_values = frame.q_H-frame.q_A if target_name=='home_minus_away' else frame.q_D
        rho = float(spearmanr(frame[col],target_values).statistic)
        correlation_rows.append({'period':label,'feature':col,'odds_target':target_name,
                                 'matches':len(frame),'spearman_rho':rho})
correlations = pd.DataFrame(correlation_rows)
correlations.to_csv(OUT / "odds_spearman_correlations.csv", index=False)
display(correlations.loc[correlations.period.eq('all_test')])

indexed_scores = scores.set_index('model')
panels = [
    (['M0', 'M1', 'M2', 'M3', 'Market'], 'log_loss', 'Result prediction'),
    (['O0', 'O1', 'O2', 'O3'], 'kl_to_odds', 'Fit market probabilities'),
    (['Market', 'Qcal', 'Q0', 'Q1', 'Q2', 'Q3'], 'log_loss', 'Information beyond odds')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, (names, metric, title) in zip(axes, panels):
    values = indexed_scores.loc[names, metric].to_numpy(float)
    x = np.arange(len(names))
    ax.plot(x, values, 'o-')
    span = float(values.max() - values.min())
    pad = span * 0.12 if span else max(abs(float(values[0])) * 0.02, 1e-4)
    ax.set_ylim(float(values.min()) - pad, float(values.max()) + pad)
    ax.set_xticks(x, [f'{name}\n{value:.5f}' for name, value in zip(names, values)])
    ax.set(title=title, ylabel=metric + ' (lower is better)')
    ax.grid(axis='y', alpha=.25)
fig.tight_layout()
fig.savefig(OUT / 'baseline_comparison.png', dpi=160)
fig.savefig(OUT / 'baseline_comparison.svg')
plt.show()


## 10. 保存结果说明

**问题:** 赛果预测、赔率拟合，以及给定赔率后是否还有增量，是三个问题。放在同一张表里会把 Log Loss 和 KL 混在一起。

**解决办法:** `baseline.md` 分开记录这三部分。结果只针对本次样本、固定 5 场、全部历史交锋和逻辑回归，不能直接推广为 LSTM 或其他窗口的结果。

本节写出 `reports/baseline/baseline.md`。

In [ ]:
def md_table(frame):
    columns = list(frame.columns)
    lines = ['| ' + ' | '.join(columns) + ' |', '| ' + ' | '.join(['---'] * len(columns)) + ' |']
    for record in frame.itertuples(index=False, name=None):
        values = []
        for value in record:
            if isinstance(value, (float, np.floating)):
                values.append(f'{value:.6f}')
            elif isinstance(value, pd.Timestamp):
                values.append(value.strftime('%Y-%m-%d'))
            else:
                values.append(str(value))
        lines.append('| ' + ' | '.join(values) + ' |')
    return '\n'.join(lines)


def comparison_row(left, right, metric):
    return paired.loc[(paired.left == left) & (paired.right == right) & (paired.metric == metric)].iloc[0]


def comparison_text(left, right, metric):
    row = comparison_row(left, right, metric)
    name = 'Log Loss' if metric == 'log_loss' else 'KL'
    direction = '更低' if row.difference < 0 else '更高'
    stability = '区间包含 0，不能认为差异稳定' if row.ci_includes_zero else '描述性区间不包含 0'
    return (f'{left} 相对 {right} 的平均 {name} {direction}，差值 {row.difference:.6f}，'
            f'95% 区间 [{row.ci_low:.6f}, {row.ci_high:.6f}]；{stability}。')


main_scores = scores.loc[scores.model.isin(['M0', 'M1', 'M2', 'M3', 'Market']), ['model', 'matches', 'log_loss', 'brier', 'accuracy']]
market_fit = scores.loc[scores.model.str.startswith('O'), ['model', 'matches', 'kl_to_odds', 'odds_probability_mae']]
market_increment = scores.loc[scores.model.isin(['Market', 'Qcal', 'Q0', 'Q1', 'Q2', 'Q3']), ['model', 'log_loss', 'brier', 'accuracy']]
test_corr = correlations.loc[correlations.period.eq('all_test'), ['feature', 'odds_target', 'spearman_rho']]
home_away = test_corr.loc[test_corr.odds_target.ne('q_D')]
top_corr = home_away.loc[home_away.spearman_rho.abs().idxmax()]
m_losses = main_scores.loc[main_scores.model.ne('Market'), 'log_loss']
market_loss = float(main_scores.loc[main_scores.model.eq('Market'), 'log_loss'].iloc[0])
m1_loss = float(main_scores.loc[main_scores.model.eq('M1'), 'log_loss'].iloc[0])
m2_loss = float(main_scores.loc[main_scores.model.eq('M2'), 'log_loss'].iloc[0])
h2h_gd_rho = float(test_corr.loc[test_corr.feature.eq('h2h_gd'), 'spearman_rho'].iloc[0])
draw_corr = test_corr.loc[test_corr.feature.str.contains('draw'), 'spearman_rho']
shown_corr = test_corr.copy()
shown_corr["spearman_rho"] = shown_corr.spearman_rho.map(lambda value: f"{value:.3f}")
recent_vs_h2h = comparison_row('M1', 'M2', 'log_loss')
odds_fit = comparison_row('O1', 'O2', 'kl_to_odds')
beyond_odds = comparison_row('Q1', 'Q0', 'log_loss')
if recent_vs_h2h.ci_includes_zero:
    result_answer = '近期状态和历史交锋对赛果都没有稳定的额外影响。'
elif recent_vs_h2h.difference < 0:
    result_answer = '近期状态比历史交锋更影响赛果。'
else:
    result_answer = '历史交锋比近期状态更影响赛果。'
if odds_fit.ci_includes_zero:
    odds_answer = '近期状态和历史交锋谁更接近盘口，描述性区间不能分开。'
elif odds_fit.difference < 0:
    odds_answer = '近期状态比历史交锋更接近盘口。'
else:
    odds_answer = '历史交锋比近期状态更接近盘口。'
if beyond_odds.ci_includes_zero:
    residual_answer = '赔率进入模型之后，再加入近 5 场，赛果 Log Loss 没有稳定下降。交锋同样没有稳定增量。'
elif beyond_odds.difference < 0:
    residual_answer = '赔率进入模型之后，近 5 场仍能稳定降低赛果 Log Loss。'
else:
    residual_answer = '赔率进入模型之后，加入近 5 场会使赛果 Log Loss 稳定升高。'

if odds_fit.ci_includes_zero:
    closer = "不能区分近 5 场和全部交锋谁更接近盘口"
elif odds_fit.difference < 0:
    closer = "近 5 场比全部交锋更接近盘口"
else:
    closer = "全部交锋比近 5 场更接近盘口"
q2 = comparison_row("Q2", "Q0", "log_loss")
if beyond_odds.ci_includes_zero and q2.ci_includes_zero:
    extra = "在预测赛果时，两者都没有提供赔率之外的稳定信息"
else:
    extra = "赔率之外是否还有增量，以上一句为准"
closing = f"因此，{closer}；{extra}。"

report = f"""# Baseline

**原题。** For a given matchup between teams, what is more useful to predict results: most recent form, or historical form between the teams. How does this correlate with the betting odds for a given match?

对一场给定对阵，预测赛果时更有用的是各队近 5 场，还是两队此前的全部交锋？这两块信息和该场赔率如何相关？

## 对原题的回答

**赛果。** {result_answer} 近 5 场（M1）的 Log Loss 是 {m1_loss:.6f}，全部交锋（M2）是 {m2_loss:.6f}。{comparison_text('M1', 'M2', 'log_loss')} 直接使用赔率的 Market 是 {market_loss:.6f}，对赛果的影响大于近期状态和历史交锋彼此的差别。

**赔率。** {odds_answer} {comparison_text('O1', 'O2', 'kl_to_odds')} 与 \\(q_H-q_A\\) 最一致的特征是 `{top_corr.feature}`，Spearman 系数为 {top_corr.spearman_rho:.3f}；交锋净胜球 `h2h_gd` 为 {h2h_gd_rho:.3f}。平局比例与 \\(q_D\\) 的相关在 {draw_corr.min():.3f} 到 {draw_corr.max():.3f} 之间。{residual_answer}

## 样本

{md_table(split_table)}

历史来自 {len(valid_history):,} 场有效赛果，预测样本来自 cleaned.csv。最终拟合使用截至 2022 年末的 {len(final_train):,} 场，评价 2023—2024 的 {len(test):,} 场。2025 及以后不拟合、不评分。

## 哪一块更能预测赛果

M0 只用基础信息 B（联赛、Elo 差、交锋场数、有无交锋）。M1 加入近 5 场，M2 加入全部交锋，M3 两块都加。Market 不训练，直接使用归一化赔率概率。Log Loss 和 Brier 越小越好。

{md_table(main_scores)}

{comparison_text('M1', 'M0', 'log_loss')}

{comparison_text('M2', 'M0', 'log_loss')}

{comparison_text('M1', 'M2', 'log_loss')}

{comparison_text('M3', 'M1', 'log_loss')}

{comparison_text('M3', 'M2', 'log_loss')}

Market 的 Log Loss 比最好的 M 模型低 {m_losses.min() - market_loss:.6f}。这个差距大于上面 M 模型彼此之间的差值。

## 哪一块更接近赔率

O0–O3 的输入与 M0–M3 相同，训练目标改为赔率概率，输入中不含赔率。KL 越小，预测分布越接近赔率。

{md_table(market_fit)}

{comparison_text('O1', 'O2', 'kl_to_odds')}

这是统计上的贴近，不能说明博彩公司实际使用了这些变量。

测试集 Spearman 相关如下。方向性特征对照 \\(q_H-q_A\\)，平局比例对照 \\(q_D\\)。相关没有控制 Elo 和联赛。

{md_table(shown_corr)}

## 已知赔率之后是否还有增量

Qcal 只用赔率对数比预测赛果。Q0 在赔率上加入 B，Q1 再加入近 5 场，Q2 再加入交锋，Q3 两块都加。

{md_table(market_increment)}

{comparison_text('Q1', 'Q0', 'log_loss')}

{comparison_text('Q2', 'Q0', 'log_loss')}

{comparison_text('Q3', 'Q0', 'log_loss')}

把原始赔率做一次校准，就能得到这组里的主要下降；近 5 场和交锋没有再把损失稳定压低。

## 图与限制

三张图都使用连续纵轴，数值写在类别名下方。左图是赛果 Log Loss，中图是赔率 KL，右图是已知赔率后的赛果 Log Loss。

![三组对照](baseline_comparison.png)

- 样本限于 Elo、近 5 场和赔率都完整的比赛，历史只覆盖文件内的赛事。
- 近 5 场等权、交锋使用此前全部场次，结论不能直接推广到时间衰减或 LSTM。
- 28 天块 bootstrap 重复 1000 次并按测试年分层。区间不含重新训练和选择 C 的不确定性。
- 预测损失下降不是投注收益。
"""
(OUT / 'baseline.md').write_text(report, encoding='utf-8')
print(comparison_text('M1', 'M2', 'log_loss'))
print(comparison_text('O1', 'O2', 'kl_to_odds'))
print('Output file:', OUT / 'baseline.md')